In [ ]:
import logging
from utils.clients import get_client
from utils.filelists import get_file_list
from dask.distributed import LocalCluster, Client, progress
import pandas as pd

### Settings

In [ ]:
CLUSTER_TYPE = "LOCAL" # either "DIRAC", "NERSC", or "LOCAL"
N_WORKERS = 5 # only applicable for DIRAC and LOCAL cluster
FILES = ""
RUN_TYPE = "CPU" # either "NUMPY", "CPU" or "GPU"
LOG_LEVEL = "INFO" # either "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"
N_FILES_FRACTION = 0.8 # Any value between 0 and 1

logging.getLogger("cabinetry").setLevel(LOG_LEVEL)

### Setup the client

In [ ]:
client = get_client(client_type=CLUSTER_TYPE, scaling=N_WORKERS)

### Filelists

In [ ]:
files = get_file_list(CLUSTER_TYPE, N_FILES_FRACTION)

### Setup the processing

In [ ]:
if RUN_TYPE == "CPU":
    from utils.numba_cpu import process_file
elif RUN_TYPE == "GPU":
    from utils.numba_cuda import process_file

In [ ]:
delayed_results = [dask.delayed(process_file)(file) for file in files]
futures = client.compute(delayed_results)
progress(futures)

### Handle results

In [ ]:
results = client.gather(futures)
results_df = pd.DataFrame(
    results,
    columns=[
        "Source",
        "nSS",
        "nSS FV",
        "nSS ROI",
        "nSS FV ROI",
        "nMSSI",
        "nMSSI FV",
        "nMSSI ROI",
        "nMSSI FV ROI"
    ],
)
results_df